# Classification de texte avec des SVM

Dans ces travaux pratiques, nous utilisons des SVM pour retrouver l'auteur d'un texte : une représentation TF-IDF et un SVM linéaire, comme sur la diapositive « SVM et texte ».

Le corpus est tiré d'œuvres classiques de la littérature anglophone : 9 auteurs, 2 livres par auteur, un fichier par chapitre.

1. Lire le corpus et le représenter en TF-IDF.
2. Évaluer un SVM linéaire, puis `LinearSVC` en réglant `C`.
3. Comprendre pourquoi ces scores sont trompeurs, et évaluer par livre.
4. Tester deux autres hypothèses : la longueur des textes et les noms propres.
5. Lire les erreurs du modèle et les mots qui le font décider.

Commençons par récupérer le corpus, et le modèle spaCy qui servira plus loin :

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm

In [ ]:
import pathlib
import time
from collections.abc import Iterable
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import sklearn.feature_extraction.text
import sklearn.metrics
import sklearn.model_selection
import sklearn.pipeline
import sklearn.svm
import spacy
import tqdm.notebook

## Création de la base d'apprentissage

Après un rapide coup d'œil aux fichiers du corpus (`classical-authors/<auteur>/<livre>/<chapitre>.txt`), écrivez la fonction `get_data`, qui prend le chemin du corpus et renvoie trois listes de même longueur, un élément par chapitre :

- les textes ;
- leur auteur ;
- leur livre (il servira plus loin).

Vous devez obtenir 1 214 chapitres, 9 auteurs et 18 livres.

**Documentation utile**

- [`pathlib.Path.glob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob)
- [`pathlib.Path.read_text`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)
- [`pathlib.PurePath.parent`](https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.parent)

In [ ]:
def get_data(directory: pathlib.Path) -> tuple[list[str], list[str], list[str]]:
  texts = []
  authors = []
  books = []

  # Votre code ici

  return texts, authors, books


texts, authors, books = get_data(pathlib.Path("classical-authors"))
print(
  len(texts), "chapitres,", len(set(authors)), "auteurs,", len(set(books)), "livres"
)

### Conseils supplémentaires

Les appels à utiliser :

- `directory.glob("*/*/*.txt")` : tous les chapitres, trois niveaux sous le corpus (`sorted` les met dans le même ordre sur toutes les machines) ;
- pour un chapitre `path`, `path.parent.name` est son livre et `path.parent.parent.name` son auteur ;
- `path.read_text(encoding="utf8")` lit tout le fichier.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def get_data(directory: pathlib.Path) -> tuple[list[str], list[str], list[str]]:
  texts = []
  authors = []
  books = []
  for path in sorted(directory.glob(...)):
    texts.append(...)
    authors.append(...)
    books.append(...)
  return texts, authors, books
```

### Solution

In [ ]:
def get_data(directory: pathlib.Path) -> tuple[list[str], list[str], list[str]]:
  texts = []
  authors = []
  books = []
  for path in sorted(directory.glob("*/*/*.txt")):
    texts.append(path.read_text(encoding="utf8"))
    authors.append(path.parent.parent.name)
    books.append(path.parent.name)
    # Variante : path.parts[-3] et path.parts[-2]
  return texts, authors, books


texts, authors, books = get_data(pathlib.Path("classical-authors"))
print(
  len(texts), "chapitres,", len(set(authors)), "auteurs,", len(set(books)), "livres"
)

## Représentation TF-IDF

Transformez `texts` en une matrice TF-IDF `X` avec le `TfidfVectorizer` de scikit-learn : une ligne par chapitre, une colonne par mot du vocabulaire. Affichez sa forme et la part de ses valeurs non nulles. Vous devez obtenir une forme de `(1214, 39623)` et 1,4 % de valeurs non nulles.

**Documentation utile**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`vectorizer.fit_transform(texts)` apprend le vocabulaire et renvoie une matrice creuse : `X.shape` donne sa forme, `X.nnz` son nombre de valeurs non nulles.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(...)
print(X.shape)
print(X.nnz / ...)
```

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(texts)
print(X.shape)
print(f"{X.nnz / (X.shape[0] * X.shape[1]):.1%} de valeurs non nulles")

## Apprentissage et évaluation

Évaluez un `SVC` à noyau linéaire par validation croisée à 5 plis (`cross_val_score`), sur `X` et `authors` : les classifieurs de scikit-learn acceptent des étiquettes texte, inutile de les encoder. Affichez les scores et le temps de calcul.

**Documentation utile**

- [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
- [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `sklearn.svm.SVC(kernel="linear")` : le SVM à noyau linéaire ;
- `cross_val_score(model, X, authors, cv=5)` renvoie un score (l'exactitude) par pli ;
- `time.perf_counter()`, appelé avant et après, mesure la durée en secondes.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
start = time.perf_counter()
scores = sklearn.model_selection.cross_val_score(..., X, authors, cv=...)
print(time.perf_counter() - start)
print(scores, scores.mean())
```

### Solution

In [ ]:
start = time.perf_counter()
scores = sklearn.model_selection.cross_val_score(
  sklearn.svm.SVC(kernel="linear"), X, authors, cv=5
)
print(f"Validation croisée : {time.perf_counter() - start:.1f} s")
print(scores, scores.mean())

## `LinearSVC` et réglage de `C`

`LinearSVC` n'est pas une implémentation plus rapide du même modèle, mais un modèle très proche, optimisé pour le noyau linéaire (carré de la perte charnière, un contre tous : voir la démo `svm-basics`, « SVM multi-classes »). Son réglage principal est `C`, la marge souple.

- Le vectoriseur, ajusté plus haut sur tous les textes, a vu les plis de test (leurs mots comptent dans l'IDF). Mettez-le avec le classifieur dans une *pipeline* (`make_pipeline`) : elle est réajustée sur les seuls plis d'apprentissage, à chaque pli.
- Réglez `C` par une recherche sur grille à 5 plis (`GridSearchCV`) sur les valeurs `np.logspace(-2, 2, 5)`.

Affichez le score de chaque `C`, le meilleur, et le temps de la recherche : comparez-le à celui du `SVC`, pour 5 fois plus d'apprentissages.

**Documentation utile**

- [`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html)
- [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `make_pipeline(TfidfVectorizer(), LinearSVC())` nomme chaque étape d'après sa classe, en minuscules : le paramètre de la grille s'écrit `"linearsvc__C"` ;
- la pipeline prend les textes bruts : `grid.fit(texts, authors)` ;
- `grid.cv_results_["mean_test_score"]` donne le score moyen de chaque `C`, `grid.best_params_` et `grid.best_score_` le meilleur ;
- `n_jobs=-1` répartit les apprentissages sur tous les cœurs.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
pipeline = sklearn.pipeline.make_pipeline(
  sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
)
grid = sklearn.model_selection.GridSearchCV(
  pipeline, {"linearsvc__C": ...}, cv=5, n_jobs=-1
)
grid.fit(..., ...)
print(grid.cv_results_["mean_test_score"])
print(grid.best_params_, grid.best_score_)
```

### Solution

In [ ]:
pipeline = sklearn.pipeline.make_pipeline(
  sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
)
grid = sklearn.model_selection.GridSearchCV(
  pipeline, {"linearsvc__C": np.logspace(-2, 2, 5)}, cv=5, n_jobs=-1
)
start = time.perf_counter()
grid.fit(texts, authors)
print(f"Recherche sur grille : {time.perf_counter() - start:.1f} s")
print(grid.cv_results_["mean_test_score"])
print(grid.best_params_, grid.best_score_)

## Conclusions

Que peut-on conclure sur les scores obtenus ?

### Solution

Des scores aussi proches de 1 sont suspects. Trois hypothèses :

- les chapitres d'un même livre sont à la fois dans les plis d'apprentissage et de test : le modèle peut reconnaître le livre (ses personnages, ses lieux, son sujet) plutôt que l'auteur ;
- les textes sont longs (des chapitres entiers) : quelques mots propres à l'auteur suffisent ;
- les noms de personnages et de lieux, cités dans presque tous les chapitres, trahissent le livre.

Pour les tester : évaluer en apprenant sur un livre de chaque auteur et en testant sur l'autre ; évaluer au niveau de la phrase ; remplacer les entités nommées.

## Évaluer par livre

Apprenez sur un livre de chaque auteur et testez sur l'autre : c'est la situation réelle, attribuer un texte que le modèle n'a jamais vu, et non un autre chapitre d'un livre qu'il connaît.

- Écrivez une fonction `get_book_index(authors, books)` qui donne à chaque chapitre le numéro de son livre chez son auteur, 0 ou 1.
- Évaluez la pipeline par `cross_val_score` avec `GroupKFold(n_splits=2)` et ces numéros comme groupes : un pli apprend sur les livres 0 et teste sur les livres 1, l'autre fait l'inverse.

Que devient le score ? Que mesurait la validation croisée à 5 plis ?

**Documentation utile**

- [`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Pourquoi pas `groups=books` ? `GroupKFold(n_splits=2)` répartirait les 18 livres en deux paquets de tailles proches, sans regarder les auteurs : les deux livres d'un même auteur peuvent tomber dans le même pli, et cet auteur n'est alors jamais vu à l'apprentissage (score autour de 0,41).
- Le premier livre d'un auteur, par ordre alphabétique : parcourez `sorted(set(zip(authors, books)))` et gardez le premier livre vu pour chaque auteur (`dict.setdefault`).
- `cross_val_score(pipeline, texts, authors, cv=..., groups=...)` : la pipeline de la partie précédente, avec `C` à sa valeur par défaut.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def get_book_index(authors: list[str], books: list[str]) -> list[int]:
  first_book = {}
  for author, book in sorted(set(zip(authors, books))):
    first_book.setdefault(author, book)
  return [... for author, book in zip(authors, books)]


book_index = get_book_index(authors, books)
scores = sklearn.model_selection.cross_val_score(
  pipeline,
  texts,
  authors,
  cv=sklearn.model_selection.GroupKFold(n_splits=...),
  groups=...,
)
print(scores, scores.mean())
```

### Solution

In [ ]:
def get_book_index(authors: list[str], books: list[str]) -> list[int]:
  first_book = {}
  for author, book in sorted(set(zip(authors, books))):
    first_book.setdefault(author, book)
  return [int(book != first_book[author]) for author, book in zip(authors, books)]


book_index = get_book_index(authors, books)
scores = sklearn.model_selection.cross_val_score(
  pipeline,
  texts,
  authors,
  cv=sklearn.model_selection.GroupKFold(n_splits=2),
  groups=book_index,
)
print(f"Par livre : {scores} {scores.mean():.3f}")

Le score tombe de 0,99 à 0,80 : la validation croisée à 5 plis mêlait les chapitres d'un même livre entre apprentissage et test, et mesurait surtout la reconnaissance d'un livre déjà vu. 0,80 est le score à attendre sur un livre inconnu, toujours loin au-dessus du hasard (1 chance sur 9). Le `C` choisi plus haut l'a été sur ces mêmes plis mêlés : il se règle lui aussi par livre.

## Régler la représentation

Variez les réglages de `TfidfVectorizer` dans la pipeline et comparez les scores par livre :

- `sublinear_tf=True` : $1 + \log(\text{tf})$ au lieu de la fréquence brute ;
- `min_df=2` : ignore les mots d'un seul chapitre ;
- `ngram_range=(1, 2)` : ajoute les paires de mots consécutifs (plus lent).

Lequel aide ? Pourquoi, selon vous ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une boucle sur des dictionnaires de paramètres, passés par `TfidfVectorizer(**params)` ; `{}` garde les réglages par défaut, pour comparer.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for params in [{}, {"sublinear_tf": True}, ...]:
  pipeline = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(**params), sklearn.svm.LinearSVC()
  )
  scores = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=book_index,
  )
  print(params, scores.mean())
```

### Solution

In [ ]:
for params in [
  {},
  {"sublinear_tf": True},
  {"min_df": 2},
  {"ngram_range": (1, 2)},
]:
  pipeline = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(**params), sklearn.svm.LinearSVC()
  )
  scores = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=book_index,
    n_jobs=-1,
  )
  print(params, f"{scores.mean():.3f}")

Par livre, `sublinear_tf=True` gagne nettement (0,80 → 0,86) : avec le logarithme, un mot répété des dizaines de fois dans un chapitre (un nom de personnage, le sujet du livre) ne l'emporte plus sur les mots que l'auteur emploie dans tous ses textes. `min_df=2` ne change presque rien (0,81) ; `ngram_range=(1, 2)` fait un peu moins bien (0,78), pour un apprentissage bien plus lent : les paires de mots ajoutent surtout des expressions propres à un livre.

## Remplacement des entités nommées par des marqueurs spéciaux

Pour remplacer les noms de personnages et de lieux (et plus généralement toutes les entités nommées), nous utilisons `spacy`, avec son modèle anglais téléchargé au début. Nous n'avons pas besoin de la plupart des composants de sa *pipeline* : nous les désactivons, et activons le seul composant supplémentaire dont nous aurons besoin (le délimiteur de phrases).

In [ ]:
nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")

Pour tester l'hypothèse des noms propres, commençons par une fonction qui remplace les entités nommées par leur type. Voici les entités que spaCy trouve dans une phrase :

In [ ]:
doc = nlp("Jimi Hendrix was absolutely amazing at Woodstock!")

# Affichage des entités nommées
print(
  " ".join(f"{t}/{t.ent_type_}/{t.ent_iob_}" if t.ent_type_ else t.text for t in doc)
)

### La fonction `replace_ners`

Si le rendu ci-dessus est :

    Jimi/PERSON/B Hendrix/PERSON/I was absolutely amazing at Woodstock/ORG/B !

la fonction `replace_ners` doit renvoyer :

    PPPERSON was absolutely amazing at OOORG !

On triple la première lettre du type : c'est une astuce pour ne pas produire un mot déjà dans le vocabulaire. Utilisez les attributs `ent_type_` et `ent_iob_` des tokens.

**Documentation utile**

- [`spacy.tokens.Token`, ses attributs](https://spacy.io/api/token#attributes)

In [ ]:
def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> str:
  # Votre code ici
  return ""


replace_ners(doc)

#### Conseils supplémentaires

- `token.ent_iob_` vaut `"B"` sur le premier token d'une entité, `"I"` sur les suivants et `"O"` hors entité : on garde le texte des `"O"`, un marqueur pour chaque `"B"`, et rien pour les `"I"` ;
- `token.ent_type_` donne le type de l'entité (`"PERSON"`, `"ORG"`…) ;
- `token.text` est le texte du token.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> str:
  result = []
  for token in tokens:
    if token.ent_iob_ == ...:
      result.append(token.text)
    elif token.ent_iob_ == ...:
      result.append(...)
  return " ".join(result)
```

#### Solution

In [ ]:
def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> str:
  result = []
  for token in tokens:
    if token.ent_iob_ == "O":
      result.append(token.text)
    elif token.ent_iob_ == "B":
      result.append(f"{token.ent_type_[0] * 2}{token.ent_type_}")
  return " ".join(result)


print(replace_ners(doc))

## Fonction `get_data` étendue

Voici la fonction `get_data` retravaillée pour renvoyer les textes à deux échelles, la phrase et le chapitre (document), chacune avec et sans remplacement des entités nommées, avec l'auteur et le livre de chaque texte. Elle renvoie un objet `Data` à 8 champs : 4 versions des textes, les auteurs et les livres aux deux échelles. Elle passe tout le corpus dans spaCy : comptez quelques minutes.

In [ ]:
class Data(NamedTuple):
  texts_sentence_replaced_ners: list[str]
  texts_sentence_normal: list[str]
  texts_document_replaced_ners: list[str]
  texts_document_normal: list[str]
  authors_sentence: list[str]
  authors_document: list[str]
  books_sentence: list[str]
  books_document: list[str]


def get_data(directory: pathlib.Path) -> Data:
  authors_sentence = []
  authors_document = []
  books_sentence = []
  books_document = []
  texts_sentence_replaced_ners = []
  texts_sentence_normal = []
  texts_document_replaced_ners = []
  texts_document_normal = []
  paths = sorted(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.notebook.tqdm(
    zip(paths, nlp.pipe(contents, n_process=1)), total=len(paths)
  ):
    author = path.parent.parent.name
    book = path.parent.name
    for sentence in doc.sents:
      authors_sentence.append(author)
      books_sentence.append(book)
      texts_sentence_replaced_ners.append(replace_ners(sentence))
      texts_sentence_normal.append(" ".join(token.text for token in sentence))
    authors_document.append(author)
    books_document.append(book)
    texts_document_replaced_ners.append(replace_ners(doc))
    texts_document_normal.append(" ".join(token.text for token in doc))
  return Data(
    texts_sentence_replaced_ners=texts_sentence_replaced_ners,
    texts_sentence_normal=texts_sentence_normal,
    texts_document_replaced_ners=texts_document_replaced_ners,
    texts_document_normal=texts_document_normal,
    authors_sentence=authors_sentence,
    authors_document=authors_document,
    books_sentence=books_sentence,
    books_document=books_document,
  )


data = get_data(pathlib.Path("classical-authors"))

## Apprentissage et évaluation des configurations

Quels sont maintenant les scores des différentes configurations (document ou phrase, avec ou sans remplacement des entités nommées), en validation croisée à 5 plis et par livre ?

Écrivez une fonction `evaluate(texts, authors, books)` qui renvoie ces deux scores moyens pour une pipeline TF-IDF + `LinearSVC`, et appelez-la sur les 4 configurations. Au niveau de la phrase, comptez quelques minutes.

Que concluez-vous sur les trois hypothèses ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les deux évaluations des parties précédentes, dans une fonction : `get_book_index(authors, books)` donne les groupes de l'évaluation par livre.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def evaluate(
  texts: list[str], authors: list[str], books: list[str]
) -> tuple[float, float]:
  pipeline = ...
  mixed = sklearn.model_selection.cross_val_score(pipeline, texts, authors, cv=5)
  by_book = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=get_book_index(..., ...),
  )
  return mixed.mean(), by_book.mean()


print(evaluate(data.texts_document_normal, data.authors_document, data.books_document))
```

### Solution

In [ ]:
def evaluate(
  texts: list[str], authors: list[str], books: list[str]
) -> tuple[float, float]:
  pipeline = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
  )
  mixed = sklearn.model_selection.cross_val_score(
    pipeline, texts, authors, cv=5, n_jobs=-1
  )
  by_book = sklearn.model_selection.cross_val_score(
    pipeline,
    texts,
    authors,
    cv=sklearn.model_selection.GroupKFold(n_splits=2),
    groups=get_book_index(authors, books),
    n_jobs=-1,
  )
  return mixed.mean(), by_book.mean()


for scale, texts_normal, texts_replaced, authors_, books_ in [
  (
    "document",
    data.texts_document_normal,
    data.texts_document_replaced_ners,
    data.authors_document,
    data.books_document,
  ),
  (
    "phrase",
    data.texts_sentence_normal,
    data.texts_sentence_replaced_ners,
    data.authors_sentence,
    data.books_sentence,
  ),
]:
  for name, texts_ in [
    ("sans remplacement des entités", texts_normal),
    ("avec remplacement des entités", texts_replaced),
  ]:
    mixed, by_book = evaluate(texts_, authors_, books_)
    print(f"{scale:<9} {name:<31} 5 plis {mixed:.3f}, par livre {by_book:.3f}")

Au niveau du document : 0,99 en 5 plis et 0,81 par livre sans remplacement des entités, 0,99 et 0,85 avec. Au niveau de la phrase : 0,64 et 0,39 sans remplacement, 0,58 et 0,40 avec.

- **Le livre** est la cause principale : par livre, le score perd près de 0,2. Les plis mêlés mesuraient surtout la reconnaissance d'un livre déjà vu.
- **La longueur** compte aussi : une phrase porte bien moins d'indices qu'un chapitre (0,39 par livre), mais reste 3 à 4 fois au-dessus du hasard (0,11) : le style se lit déjà dans quelques mots.
- **Les noms propres** n'expliquaient pas le 0,99 : les remplacer ne change presque rien en plis mêlés au niveau du document. Par livre, les remplacer *aide* (0,81 → 0,85) : les personnages d'un livre n'apparaissent pas dans l'autre, ce sont du bruit pour le modèle. En plis mêlés au niveau de la phrase, les retirer fait perdre 0,06 : un nom suffit à reconnaître le livre d'une phrase.

## Matrice de confusion

Sur les chapitres avec entités remplacées, apprenez une pipeline sur les livres 0 et prédisez l'auteur des chapitres des livres 1 (les numéros de `get_book_index`). Affichez l'exactitude, puis la matrice de confusion normalisée par ligne, avec les noms d'affichage de `DISPLAY_NAMES`.

Que lit-on sur la diagonale ? Quels auteurs le modèle confond-il ?

**Documentation utile**

- [`ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

In [ ]:
# Noms d'affichage (les dossiers du corpus gardent leurs fautes)
DISPLAY_NAMES = {
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "wilde": "Wilde",
}

# Votre code ici

### Conseils supplémentaires

- En tableaux NumPy (`np.array(..., dtype=object)` pour les textes), `texts_[book_index_ == 0]` sélectionne les chapitres des livres 0 ;
- `normalize="true"` divise chaque ligne (un auteur réel) par son total : la diagonale donne la part de ses chapitres bien attribués ;
- `labels=pipeline.classes_` fixe l'ordre des auteurs, et `display_labels` leurs noms affichés, dans le même ordre.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
texts_ = np.array(data.texts_document_replaced_ners, dtype=object)
authors_ = np.array(data.authors_document)
book_index_ = np.array(get_book_index(data.authors_document, data.books_document))

pipeline = ...
pipeline.fit(texts_[book_index_ == 0], ...)
predicted = pipeline.predict(...)

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  ...,
  predicted,
  labels=pipeline.classes_,
  display_labels=[DISPLAY_NAMES[author] for author in pipeline.classes_],
  normalize="true",
)
plt.show()
```

### Solution

In [ ]:
texts_ = np.array(data.texts_document_replaced_ners, dtype=object)
authors_ = np.array(data.authors_document)
book_index_ = np.array(get_book_index(data.authors_document, data.books_document))

pipeline = sklearn.pipeline.make_pipeline(
  sklearn.feature_extraction.text.TfidfVectorizer(), sklearn.svm.LinearSVC()
)
pipeline.fit(texts_[book_index_ == 0], authors_[book_index_ == 0])
predicted = pipeline.predict(texts_[book_index_ == 1])
print("Exactitude sur les livres 1 :", (predicted == authors_[book_index_ == 1]).mean())

sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  authors_[book_index_ == 1],
  predicted,
  labels=pipeline.classes_,
  display_labels=[DISPLAY_NAMES[author] for author in pipeline.classes_],
  normalize="true",
  values_format=".2f",
  xticks_rotation="vertical",
  cmap="Blues",
)
plt.show()

Exactitude : environ 0,85. Chaque ligne est un auteur réel, et la diagonale la part de ses chapitres bien attribués (son rappel). Les erreurs ont un sens : Wilde est pris pour Shaw dans près d'un tiers des cas (deux auteurs de théâtre : la forme dialoguée pèse plus que l'auteur), Charlotte Brontë pour sa sœur Anne (12 %), Thoreau et Emerson, deux essayistes du même cercle, se confondent un peu.

## Analyse du modèle obtenu

`LinearSVC` fait un contre tous : notre modèle est constitué de 9 SVM de classification binaire, un par auteur contre tous les autres, dans l'ordre de `classes_`.

Son attribut `coef_` contient leurs 9 vecteurs de poids, de la taille du vocabulaire : le score d'un auteur pour un document est le produit scalaire de son vecteur avec le document en TF-IDF (plus une constante). Les mots de plus grand poids sont ceux qui font le plus pencher vers cet auteur.

À l'aide de `numpy.argsort`, on affiche les mots les plus discriminants de chaque auteur face à tous les autres.

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
n_best = 10
vectorizer, model = pipeline[0], pipeline[-1]
for i, author in enumerate(model.classes_):
  author_vec = model.coef_[i]
  index = np.argsort(-author_vec)[:n_best]
  top_words = vectorizer.get_feature_names_out()[index]
  bestwords = (
    f"{word} ({score:.2f})" for score, word in zip(author_vec[index], top_words)
  )
  print(f"{DISPLAY_NAMES[author]:<17} {', '.join(bestwords)}")

Quels mots ressortent ? Qu'apprennent-ils sur ce que le modèle reconnaît ?

### Solution

Trois sortes de mots :

- des mots-outils (*the*, *his*, *of* pour Cooper ; *or*, *is* pour Emerson ; *but*, *and*, *my*, *upon* pour Anne Brontë ; *our*, *we* pour Thoreau) : le style de l'auteur, qui se retrouve d'un livre à l'autre ;
- des mots du sujet du livre (*whale*, *ship*, *sea* pour Melville ; *scout*, *rifle*, *fort* pour Cooper ; *river*, *stream* pour Thoreau) ;
- des noms propres que le petit modèle de spaCy n'a pas reconnus (*heyward*, *queequeg*, *pickering*, *higgins*, *illingworth*…), et les marqueurs eux-mêmes (*ppperson*, *ooorg*, *dddate*) : la fréquence des entités nommées est aussi un indice.

Les mots du sujet et les noms ne passent pas d'un livre à l'autre : ce sont eux qui gonflaient l'évaluation mêlée. La stylométrie, qui attribue des textes à leur auteur, s'appuie d'ailleurs surtout sur la fréquence des mots-outils.